# 5. SLDS Learning Phase Assignment

**Purpose**: Discover behavioural regimes (learning phases) from session-level
statistics, blind to session labels. Provides principled, real-time state
definitions for triggering opto experiments (Aim 2) and tracking neural
dynamics (Aim 3).

**Approach** (progressive):
1. HMM — discrete states only (baseline)
2. SLDS — discrete states × continuous dynamics (full model)

Both fitted via `ssm` (Linderman lab). Model selection by BIC over:
- K (number of discrete states)
- D (latent dimensionality, SLDS only)

**Expected states**:
- "Learning": high recency, low accuracy, unstable PSE, high choice entropy
- "Expert": low recency, high accuracy, stable PSE, low choice entropy
- Possibly: "Transition" / "Post-shift" states

**Validation**:
- Terminal expert state should look similar across animals
- SLDS states should correlate with model-based measures (η from SBI)
- Number of states by BIC, not imposed a priori

**Features from 1a**: accuracy, pse, slope, recency, choice_entropy,
win_stay, side_bias, lapse_low

```bash
pip install ssm  # git+https://github.com/lindermanlab/ssm
```

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.preprocessing import StandardScaler
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

In [ ]:
from behav_utils.data.loading import load_experiment
from behav_utils.analysis.session_features import (
    build_feature_matrix, build_feature_matrix_multi, get_feature_columns,
)
from behav_utils.plotting.styles import COLOURS, apply_style

apply_style()
print('Core imports OK')

In [ ]:
import ssm
print(f'ssm={ssm.__version__}')

## 1. Configuration

In [ ]:
CONFIG_PATH = Path('../config.yaml')
STAGE = 'Full_Task_Cont'
MIN_SESSIONS = 10

# Features from notebook 1a
SELECTED_FEATURES = [
    'accuracy', 'pse', 'slope', 'recency',
    'choice_entropy', 'win_stay', 'side_bias', 'lapse_low',
]

# Model comparison ranges
K_RANGE = range(2, 7)       # discrete states: 2–6
D_LATENT_RANGE = [1, 2, 3]  # SLDS latent dims

N_HMM_RESTARTS = 10  # EM is sensitive to initialisation
SEED = 42

## 2. Load Data & Build Feature Matrix

In [ ]:
experiment = load_experiment(CONFIG_PATH)
all_animals = experiment.get_animals(min_sessions=MIN_SESSIONS, stage=STAGE)
print(f'{len(all_animals)} animals with >= {MIN_SESSIONS} sessions')

# Build multi-animal feature matrix
df = build_feature_matrix_multi(
    all_animals, stage=STAGE,
    stat_names=['accuracy', 'psychometric', 'recency', 'choice_entropy',
                'win_stay', 'side_bias'],
)
print(f'Feature matrix: {len(df)} sessions × {len(df.columns)} columns')
print(f'Animals: {df["animal_id"].nunique()}')

In [ ]:
# Check which selected features are available
available = [f for f in SELECTED_FEATURES if f in df.columns]
missing = [f for f in SELECTED_FEATURES if f not in df.columns]
if missing:
    print(f'Missing features: {missing}')
    print(f'Available columns: {sorted(df.columns.tolist())}')
print(f'Using {len(available)} features: {available}')

FEATURES = available

In [ ]:
# NaN audit
for f in FEATURES:
    n_nan = df[f].isna().sum()
    pct = 100 * n_nan / len(df)
    print(f'  {f:>20s}: {n_nan:>4d} NaN ({pct:.1f}%)')

# Drop sessions with any NaN in selected features
df_clean = df.dropna(subset=FEATURES).copy()
print(f'\nAfter dropping NaN: {len(df_clean)} sessions '
      f'(dropped {len(df) - len(df_clean)})')

In [ ]:
# Standardise features (zero mean, unit variance) across all animals
scaler = StandardScaler()
X_all = scaler.fit_transform(df_clean[FEATURES].values)
print(f'Standardised feature matrix: {X_all.shape}')

# Split into per-animal sequences (preserving temporal order)
animal_ids = df_clean['animal_id'].unique()
animal_sequences = {}  # animal_id -> (X, session_indices)

for aid in animal_ids:
    mask = df_clean['animal_id'] == aid
    X_animal = X_all[mask.values]
    sess_idx = df_clean.loc[mask, 'session_idx'].values
    animal_sequences[aid] = (X_animal, sess_idx)
    print(f'  {aid}: {X_animal.shape[0]} sessions')

# Prepare data formats for ssm
X_list = [seq[0] for seq in animal_sequences.values()]
X_concat = np.concatenate(X_list)
n_total_obs = X_concat.shape[0]
D_obs = len(FEATURES)
print(f'\nTotal observations: {n_total_obs}, D_obs: {D_obs}')

## 3. Parameter Counting & BIC

Neither ssm HMM nor SLDS expose a built-in BIC. We count parameters
manually and compute BIC = −2·LL + n_params·ln(n_obs).

**Note on SLDS log-likelihood:** `model.log_likelihood()` computes an
approximate marginal likelihood (exact for discrete states, approximate
for continuous latent via Laplace). The approximation error is consistent
across models so relative BIC comparisons remain valid.

In [ ]:
def hmm_n_params(K: int, D: int) -> int:
    """Free parameters in ssm Gaussian-emission HMM."""
    init = K - 1                    # initial state simplex
    trans = K * (K - 1)             # transition rows (each a simplex)
    means = K * D                   # emission means
    covariances = K * D * (D + 1) // 2  # emission covariances (symmetric)
    return init + trans + means + covariances


def slds_n_params(K: int, D: int, N: int) -> int:
    """Free parameters in ssm Gaussian-emission SLDS.
    
    K: discrete states, D: latent dim, N: observation dim.
    """
    init = K - 1
    trans = K * (K - 1)
    dynamics = K * (D * D + D)              # A_k matrices + bias
    dynamics_cov = K * D * (D + 1) // 2
    emissions = K * (N * D + N)             # C_k matrices + bias
    emission_cov = K * N * (N + 1) // 2
    return init + trans + dynamics + dynamics_cov + emissions + emission_cov


def compute_bic(ll: float, n_params: int, n_obs: int) -> float:
    """BIC = -2*LL + n_params*ln(n_obs). Lower is better."""
    return -2 * ll + n_params * np.log(n_obs)


# Quick sanity check
print(f'HMM K=3, D={D_obs}: {hmm_n_params(3, D_obs)} params')
print(f'SLDS K=3, D=2, N={D_obs}: {slds_n_params(3, 2, D_obs)} params')

## 4. HMM — Discrete States

Gaussian HMM discovers discrete behavioural regimes without modelling
within-state dynamics. Pooled model (shared emission parameters, each
animal is a separate sequence). Multiple EM restarts per K.

In [ ]:
hmm_scores = {}  # K -> {ll, bic, aic, n_params, model}

for K in K_RANGE:
    best_ll = -np.inf
    best_model = None

    for restart in range(N_HMM_RESTARTS):
        model = ssm.HMM(K, D_obs, observations='gaussian')
        try:
            model.fit(X_concat, method='em', num_iters=200)
            ll = model.log_likelihood(X_concat)
            if ll > best_ll:
                best_ll = ll
                best_model = model
        except Exception:
            continue

    if best_model is not None:
        n_p = hmm_n_params(K, D_obs)
        bic = compute_bic(best_ll, n_p, n_total_obs)
        aic = -2 * best_ll + 2 * n_p
        hmm_scores[K] = {
            'll': best_ll, 'bic': bic, 'aic': aic,
            'n_params': n_p, 'model': best_model,
        }
        print(f'  K={K}: LL={best_ll:.1f}, BIC={bic:.1f}, AIC={aic:.1f} (n_params={n_p})')

if hmm_scores:
    best_K_bic = min(hmm_scores, key=lambda k: hmm_scores[k]['bic'])
    best_K_aic = min(hmm_scores, key=lambda k: hmm_scores[k]['aic'])
    print(f'\nBest K: BIC → {best_K_bic}, AIC → {best_K_aic}')

In [ ]:
# Plot model comparison
if hmm_scores:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    ks = sorted(hmm_scores.keys())

    ax = axes[0]
    ax.plot(ks, [hmm_scores[k]['ll'] for k in ks], 'o-', color='navy')
    ax.set_xlabel('Number of states (K)')
    ax.set_ylabel('Log-likelihood')
    ax.set_title('HMM: Log-likelihood')

    ax = axes[1]
    ax.plot(ks, [hmm_scores[k]['bic'] for k in ks], 'o-', color='crimson', label='BIC')
    ax.plot(ks, [hmm_scores[k]['aic'] for k in ks], 's--', color='darkorange', label='AIC')
    ax.set_ylabel('Information criterion (lower = better)')
    ax.legend()
    ax.set_xlabel('Number of states (K)')
    ax.set_title('HMM: Model selection')

    fig.suptitle('HMM State Count Selection', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

### 4b. HMM State Interpretation

In [ ]:
if hmm_scores:
    K_best = best_K_bic
    hmm_best = hmm_scores[K_best]['model']
    print(f'Using K={K_best}')

    # Decode per animal
    hmm_labels = {}
    for aid, (X_a, sess_idx) in animal_sequences.items():
        hmm_labels[aid] = hmm_best.most_likely_states(X_a)

    # State means in original (unstandardised) feature space
    # ssm Gaussian observations: params is list of (mean, cov) per state
    state_means_std = np.array([hmm_best.observations.params[k][0]
                                 for k in range(K_best)])
    state_means = scaler.inverse_transform(state_means_std)

    print(f'\nState means (original scale):')
    means_df = pd.DataFrame(state_means, columns=FEATURES,
                             index=[f'State {k}' for k in range(K_best)])
    print(means_df.round(3).to_string())

In [ ]:
# Heuristic labelling: which state is "expert"?
if hmm_scores:
    acc_col = FEATURES.index('accuracy')
    expert_state = np.argmax(state_means[:, acc_col])
    learning_state = np.argmin(state_means[:, acc_col])
    print(f'Highest accuracy: State {expert_state} '
          f'(acc={state_means[expert_state, acc_col]:.3f})')
    print(f'Lowest accuracy:  State {learning_state} '
          f'(acc={state_means[learning_state, acc_col]:.3f})')

    STATE_NAMES = {}
    STATE_NAMES[expert_state] = 'Expert'
    STATE_NAMES[learning_state] = 'Learning'
    for k in range(K_best):
        if k not in STATE_NAMES:
            STATE_NAMES[k] = f'Intermediate_{k}'

    print(f'\nState labels: {STATE_NAMES}')

In [ ]:
# Plot state trajectories per animal
if hmm_scores:
    state_colours = plt.cm.Set2(np.linspace(0, 1, K_best))

    n_animals = len(animal_sequences)
    fig, axes = plt.subplots(n_animals, 1,
                              figsize=(14, 2.5 * n_animals),
                              sharex=False)
    if n_animals == 1:
        axes = [axes]

    for ax, (aid, (X_a, sess_idx)) in zip(axes, animal_sequences.items()):
        labels = hmm_labels[aid]

        for k in range(K_best):
            mask = labels == k
            ax.scatter(sess_idx[mask], np.ones(mask.sum()) * 0.5,
                       c=[state_colours[k]], s=80, marker='s',
                       label=STATE_NAMES.get(k, f'S{k}'))

        # Overlay accuracy trajectory
        acc_idx = FEATURES.index('accuracy')
        acc_orig = scaler.inverse_transform(X_a)[:, acc_idx]
        ax2 = ax.twinx()
        ax2.plot(sess_idx, acc_orig, '-', color='grey', alpha=0.5, lw=1)
        ax2.set_ylabel('Accuracy', fontsize=8, color='grey')

        ax.set_ylabel(aid, fontsize=9, fontweight='bold')
        ax.set_yticks([])
        if ax == axes[0]:
            ax.legend(fontsize=7, loc='upper right', ncol=K_best)

    axes[-1].set_xlabel('Session index')
    fig.suptitle(f'HMM State Assignments (K={K_best})',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

### 4c. HMM Transition Matrix

In [ ]:
if hmm_scores:
    trans = np.exp(hmm_best.transitions.log_Ps)

    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(trans, cmap='Blues', vmin=0, vmax=1)
    for i in range(K_best):
        for j in range(K_best):
            ax.text(j, i, f'{trans[i, j]:.2f}', ha='center', va='center',
                    fontsize=9, color='white' if trans[i, j] > 0.5 else 'black')
    labels = [STATE_NAMES.get(k, f'S{k}') for k in range(K_best)]
    ax.set_xticks(range(K_best))
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_yticks(range(K_best))
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlabel('To')
    ax.set_ylabel('From')
    ax.set_title(f'HMM Transition Matrix (K={K_best})')
    plt.colorbar(im, ax=ax, label='P(transition)')
    plt.tight_layout()
    plt.show()

    print('\nKey transitions:')
    for i in range(K_best):
        for j in range(K_best):
            if i != j and trans[i, j] > 0.05:
                print(f'  {STATE_NAMES.get(i, f"S{i}")} → '
                      f'{STATE_NAMES.get(j, f"S{j}")}: {trans[i, j]:.3f}')

## 5. SLDS — Switching Linear Dynamical System

Extends HMM with continuous latent dynamics within each discrete state.
Grid search over K (discrete states) and D (latent dimensionality),
selected by BIC.

In [ ]:
slds_results = {}  # (K, D) -> {ll, bic, n_params, model, elbos}

for K in K_RANGE:
    for D in D_LATENT_RANGE:
        key = (K, D)
        print(f'SLDS K={K}, D={D}... ', end='', flush=True)

        model = ssm.SLDS(
            N=D_obs, K=K, D=D,
            emissions='gaussian',
            dynamics='gaussian',
        )

        try:
            elbos = model.fit(
                X_list, method='laplace_em',
                num_iters=100, initialize=True,
            )
            ll = model.log_likelihood(X_list)
            n_p = slds_n_params(K, D, D_obs)
            bic = compute_bic(ll, n_p, n_total_obs)

            slds_results[key] = {
                'll': ll, 'bic': bic,
                'n_params': n_p, 'model': model,
                'elbos': elbos,
            }
            print(f'LL={ll:.1f}  BIC={bic:.1f}  (n_params={n_p})')

        except Exception as e:
            print(f'Failed: {e}')

if slds_results:
    best_slds_key = min(slds_results, key=lambda k: slds_results[k]['bic'])
    best_K_slds, best_D_slds = best_slds_key
    print(f'\nBest SLDS by BIC: K={best_K_slds}, D={best_D_slds}')
    print(f'  BIC={slds_results[best_slds_key]["bic"]:.1f}, '
          f'LL={slds_results[best_slds_key]["ll"]:.1f}, '
          f'n_params={slds_results[best_slds_key]["n_params"]}')
else:
    print('No SLDS models fitted successfully.')
    best_K_slds, best_D_slds = None, None

In [ ]:
# SLDS BIC landscape
if slds_results:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # BIC vs K for each D
    ax = axes[0]
    for D in D_LATENT_RANGE:
        ks = [k for k in K_RANGE if (k, D) in slds_results]
        bics = [slds_results[(k, D)]['bic'] for k in ks]
        if ks:
            ax.plot(ks, bics, 'o-', label=f'D={D}')
    ax.set_xlabel('K (discrete states)')
    ax.set_ylabel('BIC (lower = better)')
    ax.set_title('SLDS: BIC vs K')
    ax.legend()

    # BIC vs D for each K
    ax = axes[1]
    for K in K_RANGE:
        ds = [d for d in D_LATENT_RANGE if (K, d) in slds_results]
        bics = [slds_results[(K, d)]['bic'] for d in ds]
        if ds:
            ax.plot(ds, bics, 'o-', label=f'K={K}')
    ax.set_xlabel('D (latent dimensions)')
    ax.set_ylabel('BIC (lower = better)')
    ax.set_title('SLDS: BIC vs D')
    ax.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
# SLDS state assignments and latent trajectories
if slds_results:
    K_slds = best_K_slds
    D_latent = best_D_slds
    slds_best = slds_results[(K_slds, D_latent)]['model']

    slds_labels = {}   # animal_id -> discrete state labels
    slds_latent = {}   # animal_id -> continuous latent trajectory

    for aid, (X_a, sess_idx) in animal_sequences.items():
        z, x = slds_best.most_likely_states(X_a)
        slds_labels[aid] = z
        slds_latent[aid] = x

    # Plot: discrete states + latent trajectory
    n_animals = len(animal_sequences)
    fig, axes = plt.subplots(n_animals, 2,
                              figsize=(16, 2.5 * n_animals))
    if n_animals == 1:
        axes = axes.reshape(1, -1)

    state_colours = plt.cm.Set2(np.linspace(0, 1, K_slds))

    for row, (aid, (X_a, sess_idx)) in enumerate(animal_sequences.items()):
        z = slds_labels[aid]
        x_lat = slds_latent[aid]

        # Discrete states
        ax = axes[row, 0]
        for k in range(K_slds):
            mask = z == k
            ax.scatter(sess_idx[mask], np.ones(mask.sum()) * 0.5,
                       c=[state_colours[k]], s=80, marker='s', label=f'S{k}')
        ax.set_ylabel(aid, fontsize=9, fontweight='bold')
        ax.set_yticks([])
        if row == 0:
            ax.legend(fontsize=7, ncol=K_slds)
            ax.set_title('Discrete states')

        # Latent dimensions
        ax = axes[row, 1]
        for d in range(D_latent):
            ax.plot(sess_idx, x_lat[:, d], '-', alpha=0.7, lw=1.2, label=f'z{d}')
        if row == 0:
            ax.legend(fontsize=7)
            ax.set_title('Continuous latent trajectory')
        ax.set_ylabel(aid, fontsize=8)

    axes[-1, 0].set_xlabel('Session')
    axes[-1, 1].set_xlabel('Session')
    fig.suptitle(f'SLDS: K={K_slds}, D={D_latent}',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 6. HMM vs SLDS Comparison

In [ ]:
if hmm_scores and slds_results:
    # BIC comparison
    hmm_bic = hmm_scores[best_K_bic]['bic']
    slds_bic = slds_results[best_slds_key]['bic']
    delta_bic = hmm_bic - slds_bic
    winner = 'SLDS' if delta_bic > 0 else 'HMM'
    print(f'HMM  (K={best_K_bic}):           BIC={hmm_bic:.1f}  '
          f'(n_params={hmm_scores[best_K_bic]["n_params"]})')
    print(f'SLDS (K={best_K_slds}, D={best_D_slds}): BIC={slds_bic:.1f}  '
          f'(n_params={slds_results[best_slds_key]["n_params"]})')
    print(f'\nΔBIC = {delta_bic:.1f} in favour of {winner}')
    if abs(delta_bic) < 10:
        print('(ΔBIC < 10: weak evidence — consider using the simpler HMM)')

    # Transition agreement
    print('\nState transition agreement (HMM vs SLDS):')
    for aid in animal_sequences:
        if aid in hmm_labels and aid in slds_labels:
            h = hmm_labels[aid]
            s = slds_labels[aid]
            h_trans = np.where(np.diff(h) != 0)[0]
            s_trans = np.where(np.diff(s) != 0)[0]
            print(f'  {aid}: HMM transitions at sessions {h_trans}, '
                  f'SLDS at {s_trans}')

## 7. Validation: Expert State Consistency

The terminal expert state should look similar across animals,
even if reached at different session numbers.

In [ ]:
if hmm_scores:
    print('Terminal state consistency (last 5 sessions):')
    terminal_states = {}
    for aid, labels in hmm_labels.items():
        last_5 = labels[-5:]
        dominant = np.bincount(last_5, minlength=K_best).argmax()
        frac = np.mean(last_5 == dominant)
        terminal_states[aid] = {
            'dominant_state': dominant,
            'state_name': STATE_NAMES.get(dominant, f'S{dominant}'),
            'fraction': frac,
        }
        print(f'  {aid}: {STATE_NAMES.get(dominant, f"S{dominant}")} '
              f'({frac:.0%} of last 5 sessions)')

    terminal_dominant = [v['dominant_state'] for v in terminal_states.values()]
    if len(set(terminal_dominant)) == 1:
        print(f'\nAll animals end in the same state: '
              f'{STATE_NAMES.get(terminal_dominant[0])}')
    else:
        print(f'\nTerminal states differ across animals: '
              f'{set(STATE_NAMES.get(s, f"S{s}") for s in terminal_dominant)}')

## 8. Online State Prediction

Given the trained HMM/SLDS, predict the current state from a new
session's features. This is the real-time trigger for Aim 2.

In [ ]:
def predict_state(session, model, scaler, features, model_type='hmm'):
    """Predict state for a single new session.
    
    Args:
        session: SessionData object
        model: trained ssm HMM or SLDS model
        scaler: fitted StandardScaler
        features: list of feature names
        model_type: 'hmm' or 'slds'
    
    Returns:
        (state_label, info_dict) or (None, None) if NaN features
    """
    stats = session.stats(features)
    x = np.array([stats.get(f, np.nan) for f in features]).reshape(1, -1)
    if np.any(np.isnan(x)):
        return None, None
    x_std = scaler.transform(x)
    
    if model_type == 'hmm':
        state = model.most_likely_states(x_std)[0]
        return state, {}
    elif model_type == 'slds':
        z, x_lat = model.most_likely_states(x_std)
        return z[0], {'latent': x_lat[0]}
    else:
        raise ValueError(f'Unknown model_type: {model_type}')


# Demo: predict state for each session of one animal
if hmm_scores:
    demo_animal = all_animals[0]
    print(f'Online prediction demo: {demo_animal.animal_id}')
    sessions = demo_animal.get_sessions(stage=STAGE)

    for sess in sessions[-5:]:
        state, info = predict_state(sess, hmm_best, scaler, FEATURES, 'hmm')
        name = STATE_NAMES.get(state, f'S{state}') if state is not None else '?'
        acc = sess.stats(['accuracy']).get('accuracy', np.nan)
        print(f'  Session {sess.session_idx}: {name} (acc={acc:.2f})')

## 9. Save Results

In [ ]:
import pickle

results_dir = Path('../results/slds')
results_dir.mkdir(parents=True, exist_ok=True)

results = {
    'features': FEATURES,
    'scaler_mean': scaler.mean_,
    'scaler_scale': scaler.scale_,
    # HMM
    'hmm_K_best': best_K_bic if hmm_scores else None,
    'hmm_state_names': STATE_NAMES if hmm_scores else {},
    'hmm_labels': hmm_labels if hmm_scores else {},
    'hmm_state_means': means_df if hmm_scores else None,
    'hmm_bic_scores': {k: v['bic'] for k, v in hmm_scores.items()} if hmm_scores else {},
}

if slds_results:
    results['slds_K_best'] = best_K_slds
    results['slds_D_best'] = best_D_slds
    results['slds_labels'] = slds_labels
    results['slds_latent'] = slds_latent
    results['slds_bic_scores'] = {str(k): v['bic'] for k, v in slds_results.items()}

# Save models separately (they can be large)
if hmm_scores:
    results['hmm_model'] = hmm_best
if slds_results:
    results['slds_model'] = slds_best

with open(results_dir / 'slds_results.pkl', 'wb') as f:
    pickle.dump(results, f)
print(f'Saved to {results_dir / "slds_results.pkl"}')

## Interpretation & Next Steps

**If HMM finds clean Learning → Expert transition:**
The transition session is the natural trigger for opto experiments.
Animals that reach Expert state for N consecutive sessions are ready
for distribution shift.

**If SLDS adds value over HMM (ΔBIC > 10):**
The continuous latent trajectory captures gradual within-state changes
that HMM misses. This is particularly valuable for the post-shift
re-learning trajectory.

**If ΔBIC < 10 (weak preference):**
Prefer the simpler HMM for online state prediction — fewer parameters,
more robust to noise, and the discrete-only representation is sufficient
for triggering opto.

**If states don't separate cleanly:**
May need more features, or a different feature set. The features from
1a were selected on pre-shift data — post-shift data may require
different features.

**Online application (Aim 2):**
Train on the Aim 1 cohort → apply `predict_state()` in real-time
to new animals → trigger opto when Expert state is stable.

**Connection to model-based measures:**
Compare SLDS states with η from SBI (notebook 3b). If SLDS "Learning"
states have higher η than "Expert" states, model-free and model-based
characterisations converge.

**Note on SLDS log-likelihood approximation:**
`ssm`'s `model.log_likelihood()` uses a Laplace approximation for the
continuous latent states. This is standard practice and the approximation
error is consistent across model configurations, so relative BIC
comparisons are valid even if absolute LL values are approximate.